In [1]:
import os
from openai import OpenAI

from NegoLib.Entities.Agents     import BUDGET_MERCHANT, LOCAL_SUPPLIER
from NegoLib.Entities.scenarios  import KIOSK_COLA, OFFICE_LOBBY
from NegoLib.Entities.utility    import merchant_utility, supplier_utility
from NegoLib.prompt_ensemble import (
    build_merchant_prompt,
    build_supplier_prompt,
    build_judge_prompt,
    Agent_call,
    Judge_call,
)

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
)


MODEL     = "gpt-5.4-mini"
merchant  = BUDGET_MERCHANT
supplier  = LOCAL_SUPPLIER
scenarios  = [KIOSK_COLA, OFFICE_LOBBY]
STRATEGY_M = "cooperative"
STRATEGY_S = "cooperative"
MAX_ROUNDS = min(merchant.max_rounds, supplier.max_rounds)
N_REPEATS  = 3

In [2]:
for scenario in scenarios:


    judge_system = build_judge_prompt(scenario)

    for repeat in range(1, N_REPEATS + 1):
        print('=' * 70)
        print(f'REPEAT {repeat} / {N_REPEATS}')
        print(f'Scenario : {scenario.name}')
        print(f'Merchant : {merchant.name}  |  Strategy: {STRATEGY_M}')
        print(f'Supplier : {supplier.name}  |  Strategy: {STRATEGY_S}')
        print('=' * 70)

        history     = []
        deal_closed = False

        for round_num in range(1, MAX_ROUNDS + 1):
            print(f'\n--- Round {round_num} ---')

            # Merchant turn
            m_system = build_merchant_prompt(
                merchant=merchant, scenario=scenario,
                strategy=STRATEGY_M, utility=merchant_utility,
                current_round=round_num,
            )
            m_answer, m_reasoning = Agent_call(
                client, MODEL, m_system, history, opening=(round_num == 1)
            )
            history.append({'speaker': 'MERCHANT', 'content': m_answer})

            # if m_reasoning:
            #     print(f'[MERCHANT reasoning]:\n{m_reasoning}\n')
            print(f'[MERCHANT]: {m_answer}\n')

            # Supplier turn
            s_system = build_supplier_prompt(
                supplier=supplier, scenario=scenario,
                strategy=STRATEGY_S, utility=supplier_utility,
                current_round=round_num,
            )
            s_answer, s_reasoning = Agent_call(
                client, MODEL, s_system, history
            )
            history.append({'speaker': 'SUPPLIER', 'content': s_answer})

            # if s_reasoning:
            #     print(f'[SUPPLIER reasoning]:\n{s_reasoning}\n')
            print(f'[SUPPLIER]: {s_answer}\n')

            # Judge turn
            j_answer, j_reasoning = Judge_call(client, MODEL, judge_system, history)

            # if j_reasoning:
            #     print(f'[JUDGE reasoning]:\n{j_reasoning}\n')
            print(f'[JUDGE]:\n{j_answer}\n')

            if 'DEAL_CLOSED' in j_answer:
                deal_closed = True
                break

        if deal_closed:
            print(f'>> Repeat {repeat}: DEAL CLOSED')
        else:
            print(f'>> Repeat {repeat}: NO DEAL — round limit reached')
        print()

# 1 min

REPEAT 1 / 3
Scenario : Street kiosk — Cola
Merchant : Budget operator  |  Strategy: cooperative
Supplier : Local distributor  |  Strategy: cooperative

--- Round 1 ---
[MERCHANT]: Cola (330ml): $0.95 per unit, 60 units

[SUPPLIER]: Cola (330ml): $1.28 per unit, 60 units

I can support that quantity and keep the terms straightforward. If you’d like, I’m also open to discussing a small adjustment for a larger order.

[JUDGE]:
STATUS: NEGOTIATION_ONGOING
MERCHANT_LATEST_OFFERS:
  Cola (330ml): $0.95 per unit, 60 units
SUPPLIER_LATEST_OFFERS:
  Cola (330ml): $1.28 per unit, 60 units
SUMMARY: The parties have exchanged initial offers, but they have not explicitly agreed on the same price.


--- Round 2 ---
[MERCHANT]: I’m prepared to move forward on a fair middle ground for this stock-up.

Cola (330ml): $1.12 per unit, 60 units

[SUPPLIER]: I can meet you on a fair middle ground and keep the order simple.

Cola (330ml): $1.20 per unit, 60 units

If that works for you, we can wrap this up.
